In [1]:
import os
import sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

26/09/24 21:45:12 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/09/24 21:45:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 21:45:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


26/09/24 21:45:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/24 21:45:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
from deltalake import DeltaTable
from pyspark.sql import functions as F

GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "gold", "integrated_taxi_trips")

def read_delta(path):
    files = DeltaTable(path).file_uris()
    return spark.read.option("basePath", path).parquet(*files)

gold = read_delta(GOLD_PATH).filter(F.col("is_valid"))
gold.createOrReplaceTempView("integrated_taxi_trips")
print("Rows in Gold view:", spark.sql("SELECT COUNT(*) FROM integrated_taxi_trips").collect()[0][0])

26/09/24 21:45:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Rows in Gold view: 9554767


In [3]:
import os
import datetime
import shutil
from pyspark.sql import functions as F
from deltalake.writer import write_deltalake

PRODUCTS_DIR = os.path.join(PROJECT_ROOT, "data", "delta", "gold", "data_products")
os.makedirs(PRODUCTS_DIR, exist_ok=True)

if os.path.exists(PRODUCTS_DIR):
    shutil.rmtree(PRODUCTS_DIR)
    print(f"Cleaned existing data products directory: {PRODUCTS_DIR}")

def add_audit_metadata(df, schema_version="v1.0"):
    now = datetime.datetime.now(datetime.timezone.utc)
    return df \
        .withColumn("_created_at", F.lit(now.strftime("%Y-%m-%d %H:%M:%S"))) \
        .withColumn("_updated_at", F.lit(now.strftime("%Y-%m-%d %H:%M:%S"))) \
        .withColumn("_source_version", F.lit("integrated_taxi_trips_v1")) \
        .withColumn("_schema_version", F.lit(schema_version))

def save_delta_product(df, path):
    # Convert PySpark DataFrame to PyArrow Table and write native Delta format
    arrow_table = df.toPandas()
    write_deltalake(path, arrow_table, mode="overwrite")

gold_df = spark.table("integrated_taxi_trips")

# 1. Daily Mobility Summary
p1_df = add_audit_metadata(
    gold_df.groupBy("event_date", "pickup_borough").agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    )
)
save_delta_product(p1_df, os.path.join(PRODUCTS_DIR, "daily_mobility_summary"))
print("Saved Product 1: daily_mobility_summary")

# 2. Taxi Zone Statistics
p2_df = add_audit_metadata(
    gold_df.groupBy("pickup_zone", "pickup_borough").agg(
        F.count("*").alias("total_pickups"),
        F.round(F.avg("fare_amount"), 2).alias("avg_pickup_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_pickup_distance")
    )
)
save_delta_product(p2_df, os.path.join(PRODUCTS_DIR, "taxi_zone_statistics"))
print("Saved Product 2: taxi_zone_statistics")

# 3. Weather Impact Summary
p3_df = add_audit_metadata(
    gold_df.groupBy("weather_condition_code").agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")
    )
)
save_delta_product(p3_df, os.path.join(PRODUCTS_DIR, "weather_impact_summary"))
print("Saved Product 3: weather_impact_summary")

# 4. Air Quality Impact Summary
p4_df = add_audit_metadata(
    gold_df.withColumn(
        "pm25_level",
        F.when(F.greatest(F.col("air_quality_pm25"), F.lit(0.0)) <= 9.0, "1: Good (0.0-9.0)")
        .when(F.col("air_quality_pm25") <= 35.4, "2: Moderate (9.1-35.4)")
        .when(F.col("air_quality_pm25") <= 55.4, "3: Unhealthy for Sensitive Groups (35.5-55.4)")
        .when(F.col("air_quality_pm25") <= 125.4, "4: Unhealthy (55.5-125.4)")
        .otherwise("5: Very Unhealthy / Hazardous")
    ).groupBy("pm25_level").agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("air_quality_pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    )
)
save_delta_product(p4_df, os.path.join(PRODUCTS_DIR, "air_quality_impact_summary"))
print("Saved Product 4: air_quality_impact_summary")

Cleaned existing data products directory: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/delta/gold/data_products


Saved Product 1: daily_mobility_summary


Saved Product 2: taxi_zone_statistics
Saved Product 3: weather_impact_summary
Saved Product 4: air_quality_impact_summary


In [4]:
import os
from IPython.display import display, Markdown

# Verify saved data products by reading them back and displaying native HTML tables
for product in sorted(os.listdir(PRODUCTS_DIR)):
    prod_path = os.path.join(PRODUCTS_DIR, product)
    if os.path.isdir(prod_path):
        df = spark.read.parquet(prod_path)
        row_count = df.count()
        pdf = df.limit(5).toPandas()
        
        # Display header
        display(Markdown(f"### Product: `{product}` *(Total Rows: {row_count})*"))
        
        # Display DataFrame natively as an HTML table
        display(pdf)

### Product: `air_quality_impact_summary` *(Total Rows: 3)*

,pm25_level,total_trips,avg_pm25,avg_distance,_created_at,_updated_at,_source_version,_schema_version
0,1: Good (0.0-9.0),9319705,4.81,3.83,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0
1,2: Moderate (9.1-35.4),233234,15.27,12.43,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0
2,3: Unhealthy for Sensitive Groups (35.5-55.4),1828,37.55,12.60,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0


### Product: `daily_mobility_summary` *(Total Rows: 723)*

,event_date,pickup_borough,total_trips,avg_fare,avg_distance,avg_duration_minutes,total_revenue,_created_at,_updated_at,_source_version,_schema_version
0,2024-01-12,Manhattan,90460,14.94,3.41,13.76,1351664.73,2026-09-24 19:45:39,2026-09-24 19:45:39,integrated_taxi_trips_v1,v1.0
1,2024-01-10,Manhattan,81784,14.23,2.35,13.04,1163439.31,2026-09-24 19:45:39,2026-09-24 19:45:39,integrated_taxi_trips_v1,v1.0
2,2024-01-05,Manhattan,94467,14.68,2.97,13.52,1386946.12,2026-09-24 19:45:39,2026-09-24 19:45:39,integrated_taxi_trips_v1,v1.0
3,2024-01-25,Queens,8245,49.85,11.93,35.63,411031.19,2026-09-24 19:45:39,2026-09-24 19:45:39,integrated_taxi_trips_v1,v1.0
4,2024-01-24,Manhattan,93393,14.99,2.83,14.88,1399709.74,2026-09-24 19:45:39,2026-09-24 19:45:39,integrated_taxi_trips_v1,v1.0


### Product: `taxi_zone_statistics` *(Total Rows: 261)*

,pickup_zone,pickup_borough,total_pickups,avg_pickup_fare,avg_pickup_distance,_created_at,_updated_at,_source_version,_schema_version
0,Upper East Side South,Manhattan,439138,12.49,1.85,2026-09-24 19:45:40,2026-09-24 19:45:40,integrated_taxi_trips_v1,v1.0
1,Midtown North,Manhattan,276218,15.40,2.62,2026-09-24 19:45:40,2026-09-24 19:45:40,integrated_taxi_trips_v1,v1.0
2,Lenox Hill West,Manhattan,229080,12.98,2.42,2026-09-24 19:45:40,2026-09-24 19:45:40,integrated_taxi_trips_v1,v1.0
3,West Village,Manhattan,228830,14.56,2.47,2026-09-24 19:45:40,2026-09-24 19:45:40,integrated_taxi_trips_v1,v1.0
4,Seaport,Manhattan,16760,21.32,8.25,2026-09-24 19:45:40,2026-09-24 19:45:40,integrated_taxi_trips_v1,v1.0


### Product: `weather_impact_summary` *(Total Rows: 14)*

,weather_condition_code,total_trips,avg_distance,avg_duration_minutes,avg_fare,_created_at,_updated_at,_source_version,_schema_version
0,12,17209,3.25,16.68,18.11,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0
1,14,143948,3.31,15.88,18.48,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0
2,13,16724,3.06,16.15,17.56,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0
3,9,402901,4.24,15.88,17.67,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0
4,5,235549,3.53,16.14,17.66,2026-09-24 19:45:41,2026-09-24 19:45:41,integrated_taxi_trips_v1,v1.0


In [21]:
import os
from IPython.display import display, Markdown

# Verify saved data products by reading them back and displaying native HTML tables
for product in sorted(os.listdir(PRODUCTS_DIR)):
    prod_path = os.path.join(PRODUCTS_DIR, product)
    if os.path.isdir(prod_path):
        df = spark.read.parquet(prod_path)
        row_count = df.count()
        pdf = df.limit(5).toPandas()
        
        # Display header
        display(Markdown(f"### Product: `{product}` *(Total Rows: {row_count})*"))
        
        # Display DataFrame natively as an HTML table
        display(pdf)

### Product: `air_quality_impact_summary` *(Total Rows: 3)*

,pm25_level,total_trips,avg_pm25,avg_distance,_created_at,_updated_at,_source_version,_schema_version
0,1: Good (0.0-9.0),9319714,4.81,3.83,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0
1,2: Moderate (9.1-35.4),233234,15.27,12.43,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0
2,3: Unhealthy for Sensitive Groups (35.5-55.4),1828,37.55,12.60,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0


### Product: `daily_mobility_summary` *(Total Rows: 728)*

,event_date,pickup_borough,total_trips,avg_fare,avg_distance,avg_duration_minutes,total_revenue,_created_at,_updated_at,_source_version,_schema_version
0,2024-01-14,Manhattan,93062,14.48,2.29,13.19,1347972.63,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
1,2024-01-24,Manhattan,93393,14.99,2.83,14.88,1399709.74,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
2,2024-01-12,Manhattan,90460,14.94,3.41,13.76,1351664.73,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
3,2024-01-10,Manhattan,81784,14.23,2.35,13.04,1163439.31,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
4,2024-01-29,Brooklyn,828,30.32,5.97,37.24,25108.85,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0


### Product: `taxi_zone_statistics` *(Total Rows: 261)*

,pickup_zone,pickup_borough,total_pickups,avg_pickup_fare,avg_pickup_distance,_created_at,_updated_at,_source_version,_schema_version
0,Gramercy,Manhattan,197046,14.02,2.75,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
1,Upper East Side South,Manhattan,439139,12.49,1.85,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
2,Lenox Hill West,Manhattan,229081,12.98,2.42,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
3,East Village,Manhattan,234955,14.64,2.87,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0
4,Flatiron,Manhattan,155188,14.29,2.42,2026-09-20 16:46:36,2026-09-20 16:46:36,integrated_taxi_trips_v1,v1.0


### Product: `weather_impact_summary` *(Total Rows: 15)*

,weather_condition_code,total_trips,avg_distance,avg_duration_minutes,avg_fare,_created_at,_updated_at,_source_version,_schema_version
0,14.0,143948,3.31,15.88,18.48,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0
1,13.0,16724,3.06,16.15,17.56,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0
2,12.0,17209,3.25,16.68,18.11,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0
3,9.0,402901,4.24,15.88,17.67,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0
4,5.0,235549,3.53,16.14,17.66,2026-09-20 16:46:37,2026-09-20 16:46:37,integrated_taxi_trips_v1,v1.0


In [ ]:
spark.stop()